<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Clave 4 · Lógica condicional legible
**Excel Corporativo · 12 Claves** — La cartera de la empresa, clasificada por reglas · ⏱ 30 min

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/04_logica_condicional.ipynb) &nbsp; [Volver al tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-04)

**Al terminar podrás:**
1. Ver cómo una regla de negocio clasifica la cartera pendiente.
2. Comparar el efecto de Y frente a O en una regla de alerta.
3. Medir lo que esconde un SI.ERROR mal usado.

---
### Cómo usar este cuaderno (no necesitas programar)
1. **Guarda tu copia:** *Archivo › Guardar una copia en Drive*. Así tus cambios no se pierden.
2. **Ejecuta todo:** *Entorno de ejecución › Ejecutar todas* (Ctrl+F9). Si Colab avisa que el cuaderno no es de Google, elige *Ejecutar de todos modos*.
3. **Juega con los controles:** las celdas con listas desplegables a la derecha se pueden cambiar. Elige una opción y pulsa el botón ▶ de esa celda para redibujar.

El código está oculto a propósito: aquí importa entender el dato y el gráfico. Si tienes curiosidad, haz doble clic en una celda para verlo.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

## 1 · La regla de Cartera
| Si los días de mora son… | La factura queda en |
|---|---|
| 0 o menos | No vencida |
| de 1 a 30 | 1–30 |
| de 31 a 60 | 31–60 |
| más de 60 | Más de 60 |

**Días de mora** = fecha de corte − fecha de vencimiento. **Fecha de vencimiento** = fecha de la factura + días de crédito del cliente. Solo cuentan las facturas en estado *Pendiente*.

In [ ]:
# @title Preparar la cartera y elegir la fecha de corte
corte = "2026-01-31"  # @param ["2026-01-31", "2025-12-31"]
p = (ventas[ventas.estado == "Pendiente"].groupby("id_factura")
     .agg(fecha=("fecha", "first"), id_cliente=("id_cliente", "first"), id_sede=("id_sede", "first"), total=("total", "sum")).reset_index()
     .merge(clientes[["id_cliente", "razon_social", "dias_credito"]], on="id_cliente").merge(sedes[["id_sede", "ciudad"]], on="id_sede"))
p["vencimiento"] = p.fecha + pd.to_timedelta(p.dias_credito, unit="D")
p["dias_mora"] = (pd.Timestamp(corte) - p.vencimiento).dt.days
RANGOS = ["No vencida", "1–30", "31–60", "Más de 60"]
def rango(d):                     # lo mismo que SI.CONJUNTO, en el mismo orden
    if d <= 0: return "No vencida"
    if d <= 30: return "1–30"
    if d <= 60: return "31–60"
    return "Más de 60"
p["rango"] = p.dias_mora.map(rango)
display(Markdown(f"**{len(p)} facturas pendientes** por **{millones(p.total.sum())}**, con corte al {pd.Timestamp(corte):%d/%m/%Y}."))

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cuánta plata está en cada rango de mora? |
> | **Qué se grafica** | Barras: valor pendiente por rango, en el orden de la regla · Etiqueta: número de facturas |
> | **Por qué esta forma** | El orden de los rangos es parte del mensaje (de menos a más grave), así que no se ordena por tamaño. |
> | **Qué se descartó** | Torta: no muestra el orden de gravedad. |
> | **Cómo se lee** | Las barras de la derecha son las que preocupan. Cambia el corte arriba y vuelve a ejecutar: la misma cartera envejece. |
> | **En Excel 365** | Tabla dinámica con rango en Filas y total en Valores, con los rangos ordenados a mano. |

In [ ]:
# @title Gráfico 1 · Cartera por rango de mora
r = p.groupby("rango").agg(valor=("total", "sum"), n=("id_factura", "size")).reindex(RANGOS).fillna(0)
colores = [AZUL, TEAL, MAGENTA, "#7A1F3D"]
fig = go.Figure(go.Bar(x=RANGOS, y=r.valor / 1e6, marker_color=colores,
                       text=[f"$ {v / 1e6:,.0f} M · {int(n)} fact.".replace(",", ".") for v, n in zip(r.valor, r.n)], textposition="outside", cliponaxis=False))
vencida = r.loc[RANGOS[1:], "valor"].sum() / r.valor.sum()
fig.update_layout(title=f"Con corte al {pd.Timestamp(corte):%d/%m/%Y}, el {porc(vencida)} de la cartera está vencida", yaxis_title=MM, height=380)
dar_aire(fig, r.valor.max() / 1e6, "y")
fig.show()
mayor = r.valor.idxmax()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué sede tiene la cartera más envejecida? |
> | **Qué se grafica** | Barras apiladas al 100 %: composición de la cartera de cada sede por rango, ordenadas por porcentaje vencido |
> | **Por qué esta forma** | Compara la composición entre sedes sin que el tamaño de Bogotá lo domine. |
> | **Qué se descartó** | Valores absolutos: Bogotá parecería la peor solo por ser la más grande. |
> | **Cómo se lee** | Mira el ancho de los colores de la derecha en cada barra. |
> | **En Excel 365** | Dinámica sede × rango con Mostrar valores como › % del total de fila. |

In [ ]:
# @title Gráfico 2 · Envejecimiento por sede
c = pd.crosstab(p.ciudad, p.rango, values=p.total, aggfunc="sum", normalize="index").reindex(columns=RANGOS).fillna(0)
c = c.loc[(1 - c["No vencida"]).sort_values().index]
fig = go.Figure([go.Bar(name=k, y=c.index, x=c[k], orientation="h", marker_color=col, marker_line_color="white", marker_line_width=2,
                        text=[f"{v:.0%}" if v > 0.06 else "" for v in c[k]]) for k, col in zip(RANGOS, colores)])
fig.update_layout(barmode="stack", xaxis_tickformat=".0%", legend_title="", height=360,
                  title=f"{c.index[-1]} tiene la cartera más envejecida: {porc(1 - c['No vencida'].iloc[-1])} vencida")
fig.show()

## 2 · Y frente a O
La regla de alerta es «Llamar hoy si la mora supera X días **y** el total supera Y pesos». Cambia el operador y los umbrales.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué facturas quedan en «Llamar hoy» con la regla elegida? |
> | **Qué se grafica** | Dispersión: un punto por factura · Eje X: días de mora · Eje Y: total · Color: alerta · Líneas: umbrales |
> | **Por qué esta forma** | Muestra la regla como una zona del gráfico: con Y es la esquina superior derecha; con O, todo lo que pasa cualquiera de las dos líneas. |
> | **Qué se descartó** | Una tabla filtrada: no se ve por qué una factura entra y otra no. |
> | **Cómo se lee** | Los puntos magenta son los que se llaman hoy. Compara cuántos hay con Y y con O. |
> | **En Excel 365** | =SI(Y([@dias_mora]>30;[@total]>5000000);"Llamar hoy";"Seguimiento normal") |

In [ ]:
# @title Gráfico 3 · La regla de alerta como zona
operador = "Y"  # @param ["Y", "O"]
dias = "30"  # @param ["15", "30", "45"]
monto_millones = "5"  # @param ["2", "5", "10"]
d, m = int(dias), int(monto_millones) * 1e6
cond = (p.dias_mora > d) & (p.total > m) if operador == "Y" else (p.dias_mora > d) | (p.total > m)
p["alerta"] = np.where(cond, "Llamar hoy", "Seguimiento normal")
fig = px.scatter(p, x="dias_mora", y=p.total / 1e6, color="alerta", hover_name="razon_social",
                 color_discrete_map={"Llamar hoy": MAGENTA, "Seguimiento normal": GRIS})
fig.add_vline(x=d, line_dash="dot"); fig.add_hline(y=m / 1e6, line_dash="dot")
fig.update_traces(marker_size=9, marker_line_color="white", marker_line_width=1)
llamar = (p.alerta == "Llamar hoy").sum()
fig.update_layout(title=f"Con «{operador}»: {llamar} facturas para llamar hoy ({millones(p.loc[p.alerta == 'Llamar hoy', 'total'].sum())})",
                  xaxis_title="Días de mora", yaxis_title="Total de la factura (millones)", legend_title="", height=420)
fig.show()

## 3 · Lo que esconde un SI.ERROR mal usado
En la vida real algunas fechas llegan mal escritas. Simulamos 8 facturas con la fecha de vencimiento dañada.

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Dónde terminan las facturas con fecha dañada según cómo se maneje el error? |
> | **Qué se grafica** | Barras agrupadas: facturas por rango, con SI.ERROR(…;0) y con SI.ERROR(…;"Revisar fecha") |
> | **Por qué esta forma** | Las dos versiones lado a lado muestran a dónde «se esconden» los errores. |
> | **Qué se descartó** | Contar errores: no muestra que el 0 los mete entre las facturas al día. |
> | **Cómo se lee** | Con 0, las dañadas inflan «No vencida». Con «Revisar fecha», aparecen como una categoría que alguien debe corregir. |
> | **En Excel 365** | =SI.ERROR($K$2-[@fecha_vencimiento];"Revisar fecha") |

In [ ]:
# @title Gráfico 4 · SI.ERROR con 0 frente a «Revisar fecha»
danadas = p.sample(8, random_state=7).index
con_cero = p.rango.copy(); con_cero[danadas] = "No vencida"
con_aviso = p.rango.copy(); con_aviso[danadas] = "Revisar fecha"
orden = RANGOS + ["Revisar fecha"]
a = con_cero.value_counts().reindex(orden).fillna(0); b = con_aviso.value_counts().reindex(orden).fillna(0)
fig = go.Figure([go.Bar(name="SI.ERROR(…;0)", x=orden, y=a, marker_color=GRIS, text=a.astype(int)),
                 go.Bar(name="SI.ERROR(…;\"Revisar fecha\")", x=orden, y=b, marker_color=AZUL, text=b.astype(int))])
fig.update_layout(barmode="group", title="Con 0, las 8 facturas dañadas se esconden entre las «No vencidas»", yaxis_title="Facturas", legend_title="", height=380)
fig.show()

## 4 · Retos de interpretación

In [ ]:
# @title Reto 1 · Con el corte que elegiste, ¿qué rango concentra más dinero?
respuesta = "Elige una opción"  # @param ["Elige una opción", "No vencida", "1–30", "31–60", "Más de 60"]
revisar(respuesta, mayor, """Es la barra más alta del gráfico 1.""")

In [ ]:
# @title Reto 2 · Si la regla de alerta usa O en lugar de Y, las facturas en «Llamar hoy»…
respuesta = "Elige una opción"  # @param ["Elige una opción", "Aumentan", "Disminuyen", "Quedan iguales"]
revisar(respuesta, "Aumentan", """Con O basta cumplir una de las dos condiciones: la zona del gráfico 3 se agranda.""")

In [ ]:
# @title Reto 3 · =SI.CONJUNTO(A2<=60; «31–60»; A2<=30; «1–30»; VERDADERO; «Más de 60») con A2 = 10 devuelve…
respuesta = "Elige una opción"  # @param ["Elige una opción", "1–30", "31–60", "Más de 60"]
revisar(respuesta, "31–60", """SI.CONJUNTO devuelve la primera prueba verdadera: el orden de las pruebas es parte de la regla.""")

## Apóyate en IA
Pídele a la IA que convierta la regla en fórmula y, sobre todo, que te dé los casos límite para probarla:

```text
Regla de Cartera: 0 o menos días de mora = "No vencida"; 1 a 30 = "1–30"; 31 a 60 = "31–60"; más de 60 = "Más de 60".
Escríbela en Excel 365 en español con SI.CONJUNTO y la columna [@dias_mora]. Dame los casos de prueba de los límites (0, 1, 30, 31, 60, 61) con el resultado esperado.
```

> **Verifica siempre.** Compara la respuesta de la IA con un dato que ya conozcas. No pegues datos personales reales (Ley 1581 de 2012): estos datos son ficticios, los de tu empresa no.

## Resumen
* Primero la regla en una tabla de decisión; después la fórmula.
* SI.CONJUNTO evalúa en orden: la primera prueba verdadera gana.
* Y estrecha la regla; O la amplía. SI.ERROR no debe esconder datos que alguien tiene que corregir.

**Siguiente:** [Clave 5 · Buscar y cruzar sin #N/D](https://omarli-360.github.io/excel-corporativo-12-claves/#clave-05) · [Cuaderno 05](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/05_busqueda_y_cruce.ipynb)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>